# Google ADK

Concierge Agents

> The opportunity for personal AI agents to streamline and simplify people’s lives is incredible. From managing the invite list for a party to planning a garden, or helping manage complicated medications - safe and secure agents can free time for things that really matter. In this track, you will solve individual, family or social challenges in a way that keeps personal information safe and secure.

In [11]:
import datetime
from zoneinfo import ZoneInfo
from google.adk.agents import LlmAgent
from google.genai import types
from pydantic import BaseModel, Field
from google.adk.runners import InMemoryRunner

In [9]:
# Define a tool function
def get_capital_city(country: str) -> str:
  """Retrieves the capital city for a given country."""
  # Replace with actual logic (e.g., API call, database lookup)
  capitals = {"france": "Paris", "japan": "Tokyo", "canada": "Ottawa"}
  return capitals.get(country.lower(), f"Sorry, I don't know the capital of {country}.")

In [10]:
# Example: Defining the basic identity
capital_agent = LlmAgent(
    model="gemini-flash-latest",
    name="capital_agent",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country.
        When a user asks for the capital of a country:
        1. Identify the country name from the user's query.
        2. Use the `get_capital_city` tool to find the capital.
        3. Respond clearly to the user, stating the capital city.
        Example Query: "What's the capital of {country}?"
        Example Response: "The capital of France is Paris."
        Any other question from the user that is different from asking the capital return "I can't answer it"
    """,
    generate_content_config=types.GenerateContentConfig(
        temperature=0.2, # More deterministic output
        max_output_tokens=100,
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            )
        ]
    )
)

In [ ]:
# 1. Initialize an in-memory runner with your agent
runner = InMemoryRunner(agent=capital_agent)

# 2. Define the user input 
user_query = "What is the capital city of Japan?"

# Prepare the prompt structure
content = types.Content(
    role="user", 
    parts=[types.Part.from_text(text=user_query)]
)

# 3. Call the agent inside the notebook cell using await
print(f"User: {user_query}\n---")

async for event in runner.run_async(
    user_id="notebook-user-123",
    session_id="session-xyz",
    new_message=content
):
    # Capture and print the finalized text response from the agent
    if event.is_final_response():
        final_text = event.content.parts[0].text
        print(f"Agent Response: {final_text}")

----------

# Card Transactions

### Transaction File Pattern

```json
{
  "transaction_id": "tx_string",
  "timestamp": "YYYY-MM-DDTHH:MM:SSZ",
  "cardholder_id": "usr_string",
  "amount": 0.00,
  "currency": "USD",
  "merchant": {
    "name": "Merchant Name",
    "category": "MCC_CODE", 
    "id": "merch_string"
  },
  "location": {
    "city": "City Name",
    "country": "2-Letter Code",
    "terminal_type": "ONLINE" | "POS_CHIP" | "POS_TAP" | "ATM"
  },
  "network": {
    "ip_address": "v4 or v6",
    "vpn_detected": true | false,
    "proxy_detected": true | false
  },
  "device": {
    "device_fingerprint": "hash_string",
    "os": "iOS" | "Android" | "Windows" | "MacOS",
    "browser_or_app": "App" | "Safari" | "Chrome"
  },
  "payment_details": {
    "cvv_validated": true | false,
    "emv_fallback": true | false
  }
}
```

### User Profiles

#### User 1: "Digital"

```Profile Baseline:``` 
> Alex is a tech-savvy heavy e-commerce user. He buys frequently from Amazon, Steam, food delivery apps, and SaaS platforms. He uses multiple devices (MacBook, iPhone), frequently switches between home Wi-Fi and mobile networks, and occasionally uses commercial VPNs.

```The Fraud Strategy:```
> Because his baseline is highly chaotic and digital, simple location or VPN flags won't catch his frauds. The frauds here rely on credential stuffing/account takeovers (abrupt device fingerprint changes buying high-value physical goods) and card testing (micro-transactions followed by immediate high-value spikes).

#### User 2: "Older gentleman"

```Profile Baseline:``` 
> Bob is an older gentleman who rarely uses credit cards. When he does, it is strictly at ATMs or physical POS terminals in his hometown of Omaha, Nebraska, to buy gas or hardware tools. He has zero historical e-commerce baseline.

```The Fraud Strategy:```
> This profile is incredibly easy for behavioral models to monitor. Any online transaction, international billing, or sudden high-volume pattern stands out instantly as data theft or a skimmed card clone.

#### User 3: "mother"

```Profile Baseline:``` 
> Clara is a mother whose spending patterns are tightly tied to predictable family maintenance. She shops at local wholesale grocers, pharmacies, clothing retail centers, and child-care merchants. Her online spending is strictly limited to known family platforms like Target or Amazon.

```The Fraud Strategy:```
> Frauds here compromise her card information to purchase items outside the family paradigm—such as high-end gaming marketplaces, international flight aggregators, or overnight electronics drops—while trying to mimic normal processing hours.

#### User 4: "Habitual guy"

```Profile Baseline:``` 
> David is a creature of absolute habit. Every single day, he swipes his card at the exact same geographic nodes: a specific Shell station near his apartment in Boston, the same Starbucks by his office, and a local organic lunch spot. He uses Apple Pay (POS_TAP) almost exclusively.

```The Fraud Strategy:```
> Frauds against David target online portals or alternative terminals where Apple Pay tokenization isn't enforced. Because David's network footprint is purely physical, any transaction missing a local terminal identifier should raise immediate red flags.

### Detecting Fraud Skill

```
behavioral_profiler_skill/
├── SKILL.md                  # Agent definition, system instructions, and tool mapping
├── data/
│   ├── usr_alex_digital.csv  # 10 normal historical tx, 1 caught fraud
│   ├── usr_bob_ghost.csv     # 10 normal historical tx, 1 caught fraud
│   ├── usr_clara_mother.csv  # 10 normal historical tx, 1 caught fraud
│   └── usr_david_creature.csv# 10 normal historical tx, 1 caught fraud
├── scripts/
│   └── profile_db_tool.py    # Mock SQLite/Pandas search tool for the ADK framework
├── reference/
│   └── behavioral_rules.md   # Supplementary domain context on fraud velocity metrics
└── assets/
    └── .gitkeep              # Destination folder for generated behavioral reports
```

In [17]:
df = pd.read_csv(
    os.path.join(os.getcwd(), 'detectingfraud/assets', f'user_1.csv')
)
df

,transaction_id,timestamp,amount,currency,merchant_name,merchant_category,location_city,location_country,terminal_type,vpn_detected,device_fingerprint,is_fraud
0,tx_h_alex_01,2026-05-15T10:00:00Z,15.99,USD,Netflix Inc,STREAMING,Seattle,US,ONLINE,False,mac_safari_991,False
1,tx_h_alex_02,2026-05-16T12:00:00Z,35.40,USD,Uber Eats,FOOD_DELIVERY,Seattle,US,ONLINE,False,iph_app_442,False
2,tx_h_alex_03,2026-05-18T19:22:00Z,120.00,USD,Amazon.com,E_COMMERCE,Seattle,US,ONLINE,False,mac_safari_991,False
3,tx_h_alex_04,2026-05-20T14:15:00Z,8.50,USD,Spotify US,STREAMING,Seattle,US,ONLINE,False,mac_safari_991,False
4,tx_h_alex_05,2026-05-22T21:00:00Z,45.00,USD,Steam Games,DIGITAL_GOODS,Seattle,US,ONLINE,True,mac_safari_991,False
5,tx_h_alex_06,2026-05-25T11:30:00Z,22.10,USD,DoorDash,FOOD_DELIVERY,Seattle,US,ONLINE,False,iph_app_442,False
6,tx_h_alex_07,2026-05-28T09:00:00Z,250.00,USD,AWS Cloud Services,SAAS,Seattle,US,ONLINE,True,mac_safari_991,False
7,tx_h_alex_08,2026-06-01T15:00:00Z,14.99,USD,Netflix Inc,STREAMING,Seattle,US,ONLINE,False,mac_safari_991,False
8,tx_h_alex_09,2026-06-05T18:45:00Z,62.30,USD,Uber Eats,FOOD_DELIVERY,Seattle,US,ONLINE,False,iph_app_442,False
9,tx_h_alex_10,2026-06-10T13:12:00Z,110.00,USD,Amazon.com,E_COMMERCE,Seattle,US,ONLINE,False,mac_safari_991,False


In [ ]:
pd.DataFrame({
    'transaction_id': [],
    'timestamp': [],
    'amount': [],
    'currency': [],
    'merchant_name': [],
    'merchant_category': [],
    'location_city': [],
    'location_country': [],
    'terminal_type': [],
    'vpn_detected': [],
    'device_fingerprint': [],
    'is_fraud': [],
})

In [1]:
import pandas as pd
from detectingfraud.scripts.profile_db_tool import get_user_behavioral_baseline

In [2]:
summary = get_user_behavioral_baseline(cardholder_id='user_1')

In [3]:
summary

'=== BEHAVIORAL PROFILE FOR user_1 ===\nHistorical Average Ticket Size: $68.43\nMaximum Historical Ticket Size: $250.00\nFrequent Merchant Verticals: FOOD_DELIVERY, STREAMING\nEstablished Safe Locations: Seattle\nUtilized Terminal Types: ONLINE\nPrior Fraud Incidents on Account: 1 detected historically.\n\n=== LAST 10 VALID TRANSACTIONS ===\n| transaction_id   | timestamp            |   amount | currency   | merchant_name      | merchant_category   | location_city   | location_country   | terminal_type   | vpn_detected   | device_fingerprint   | is_fraud   |\n|:-----------------|:---------------------|---------:|:-----------|:-------------------|:--------------------|:----------------|:-------------------|:----------------|:---------------|:---------------------|:-----------|\n| tx_h_alex_01     | 2026-05-15T10:00:00Z |    15.99 | USD        | Netflix Inc        | STREAMING           | Seattle         | US                 | ONLINE          | False          | mac_safari_991       | Fals

In [4]:
import os
from dotenv import load_dotenv

# Load the variables from the .env file into the environment
load_dotenv(os.path.join(
    os.getcwd(), '.env'
))

True

In [11]:
os.getcwd()

'/Users/samico/Documents/DOUTORADO/PPGEEC-Transformer,Agents/Projeto06/concier_agent'

In [5]:
from google.adk import Agent
from google.adk.code_executors.unsafe_local_code_executor import UnsafeLocalCodeExecutor
from google.adk.skills import load_skill_from_dir
from google.adk.skills import models
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.skill_toolset import SkillToolset
from google.genai import types

In [6]:
## Load Skill
detecting_fraud_skill = load_skill_from_dir(
    os.path.join(os.getcwd(), 'detectingfraud')
)

# WARNING: UnsafeLocalCodeExecutor has security concerns and should NOT
# be used in production environments.
my_skill_toolset = SkillToolset(
    skills=[detecting_fraud_skill],
    code_executor=UnsafeLocalCodeExecutor(),
)

#### Testing Single Skill

In [7]:
root_agent = Agent(
    model = "gemini-2.5-flash",
    name="skill_user_agent",
    description="""An agent that helps user to know the transactions made by their credit card. You have access to specialized folder-based skills
        CRITICAL: Always use the 'load_skill' tool to read the skill instructions (SKILL.md) 
        completely before attempting to use any references or script execution tools.""",
    tools=[
        my_skill_toolset,
    ],
)

In [8]:
query_example = """Here is the transaction payload:
{
    "transaction_id": "tx_alex_01",
    "timestamp": "2026-06-20T08:12:00Z",
    "cardholder_id": "user_1",
    "amount": 14.99,
    "currency": "USD",
    "merchant": { "name": "Netflix Inc", "category": "STREAMING", "id": "m_net_88" },
    "location": { "city": "Seattle", "country": "US", "terminal_type": "ONLINE" },
    "network": { "ip_address": "67.180.20.44", "vpn_detected": false, "proxy_detected": false },
    "device": { "device_fingerprint": "mac_safari_991", "os": "MacOS", "browser_or_app": "Safari" },
    "payment_details": { "cvv_validated": true, "emv_fallback": false }
}
"""

In [9]:
from google.adk.apps import App
from google.adk.runners import InMemoryRunner

In [10]:
# Wrap your root agent inside the required App container
app = App(
    name="notebook_test_app",
    root_agent=root_agent
)
# Initialize the runner with your app container
runner = InMemoryRunner(app=app)

# Run the query asynchronously
response = await runner.run_debug(query_example)

# Print out the clean markdown string response
response

skill_user_agent > I am sorry, but I encountered an error while trying to process your transaction. I was unable to load the necessary resources within the `detectingfraud` skill, specifically `references/behavioral_rules.md` and `scripts/profile_db_tool`. This prevents me from performing the behavioral assessment of the transaction.

Please check the skill's resources or try again later.


AttributeError: 'list' object has no attribute 'text'

### Explain Potential Fraud

### Skill Evaluation

#### Eval dataset

```json
[
  { "query": "I've got a spreadsheet in ~/data/q4_results.xlsx with revenue in col C and expenses in col D — can you add a profit margin column and highlight anything under 10%?", "should_trigger": true },
  { "query": "whats the quickest way to convert this json file to yaml", "should_trigger": false }
]
```

```Thumb Rule```:
> Aim for about 20 queries: 8-10 that should trigger and 8-10 that shouldn’t.

These test whether the description captures the skill’s scope. Vary them along several axes:
- **Phrasing**: some formal, some casual, some with typos or abbreviations.
- **Explicitness**: some name the skill’s domain directly (“analyze this CSV”), others describe the need without naming it (“my boss wants a chart from this data file”).
- **Detail**: mix terse prompts with context-heavy ones — a short “analyze my sales CSV and make a chart” alongside a longer message with file paths, column names, and backstory.
- **Complexity**: vary the number of steps and decision points. Include single-step tasks alongside multi-step workflows to test whether the agent can discern the skill is relevant when the task it addresses is buried in a larger chain.


> The most valuable negative test cases are near-misses — queries that share keywords or concepts with your skill but actually need something different. These test whether the description is precise, not just broad.